# ChronosDecider: The Narrative Time Weaver — Walkthrough

**Real-time agentic Game Master**: Strands Decider 2B decides the narrative branch; Amazon Bedrock narrates it.

> Submission tags: `#strands-decider-2b` `#tool-selection`

This notebook is a **walkthrough** of the repository code (modules in `src/` and `decision_patterns/`). It reimplements nothing: it imports the engine and shows the flow.

- **Without the Decider server or Bedrock configured**, the engine degrades to local backends (simulated decision + template narrative) so the demo runs anywhere.
- **With `strands-decider serve` and Bedrock configured**, the same cells use the real decision model and the generated narrative.

Run it from the **repository root** so the `src` imports work.

## 1. Environment setup

We make sure the repository root is on `sys.path` so we can import `src` and `decision_patterns`.

In [ ]:
import sys, os

# If the notebook runs from notebooks/, go up one level to the repo root.
repo_root = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

print("Repo root:", repo_root)

## 2. The decision (Strands Decider 2B) and the generation (Bedrock)

The `DecisionEngine` orchestrates everything: it decides the branch with Decider 2B, applies the confidence triage (`<= 0.70` → Human-in-the-Loop) and, if the decision is reliable, narrates with Bedrock.

In [ ]:
from src.engine import DecisionEngine

engine = DecisionEngine()

def show(result):
    print(f"Classified action : {result.action_type}")
    print(f"Narrative branch  : {result.target_branch}")
    print(f"Confidence        : {result.confidence_score:.1%}")
    print(f"Decision latency  : {result.decision_latency_ms:.2f} ms (target < 100 ms, within = {result.within_latency_target})")
    print(f"Decision via      : {result.decision_source}")
    print(f"Narrative via     : {result.narrative_source}")
    print(f"Fallback          : {result.trigger_fallback}")
    print(f"GM response       : {result.narrative_response or '(not narrated: clarification requested)'}")
    if result.clarification_prompt:
        print(f"Clarification     : {result.clarification_prompt}")

### 2a. Clear action → high confidence → it narrates

Decider picks a branch with confidence above the threshold, so the agent acts and Bedrock (or the template) narrates.

In [ ]:
show(engine.evaluate("attack the goblin with my sword"))

### 2b. Ambiguous action → low confidence → guardrail (Human-in-the-Loop)

Here is the value of **calibrated confidence**: when Decider is unsure, the agent does NOT invent a narration; it asks for clarification.

In [ ]:
show(engine.evaluate("hmm I don't know what to do"))

## 3. The reusable decision pattern (`#tool-selection`)

The hackathon's reusable deliverable lives in `decision_patterns/`. It is independent of the game: it builds a Decider `choice` question and returns the choice with its confidence, plus a `should_act` flag that applies the threshold as a guardrail.

The cells below show the pattern's **pure functions** (building the question and parsing the response) without needing a running server.

In [ ]:
from decision_patterns import build_choice_question, parse_choice_answer

# 1) The typed question sent to Strands Decider 2B.
question = build_choice_question(
    "Which team should handle this message?",
    ["billing", "sales", "retail"],
)
question

In [ ]:
# 2) Parsing a server response (example): choice + calibrated confidence.
server_response = {
    "answers": {"tool_choice": {"choice": "billing", "confidence": 0.83}},
    "latency_ms": 118.0,
}
choice = parse_choice_answer(
    server_response, ["billing", "sales", "retail"], threshold=0.70, latency_ms=120.0
)
print("Chosen tool :", choice.tool)
print("Confidence  :", choice.confidence)
print("Should act? :", choice.should_act, "(if False, the agent asks for clarification)")

### 3a. (Optional) Against the real Strands Decider server

If you have the server running:

```bash
pip install strands-decider
strands-decider serve StrandsAgents/strands-decider-2B-qwen3.5-v1-2610 --port 8000
```

the next cell makes a **real** decision. If the server does not respond, it reports it without breaking the notebook.

In [ ]:
import urllib.error
from decision_patterns import ToolSelectionPattern

pattern = ToolSelectionPattern(
    instructions="Which team should handle this message?",
    options=["billing", "sales", "retail"],
    confidence_threshold=0.70,
)

try:
    result = pattern.select("Help! My payouts have been failing for 3 days.")
    print("Real choice:", result.tool, "| confidence:", round(result.confidence, 3),
          "| latency:", round(result.latency_ms, 1), "ms")
except (urllib.error.URLError, ConnectionError, TimeoutError, OSError) as exc:
    print("Strands Decider server not available:", exc)
    print("Start it with: strands-decider serve StrandsAgents/strands-decider-2B-qwen3.5-v1-2610 --port 8000")

## 4. Telemetry

The engine records, for each decision, the latency and the confidence. Aggregated summary after the evaluations above:

In [ ]:
summary = engine.telemetry.summary()
if summary is None:
    print("No telemetry records yet.")
else:
    print(f"Evaluations      : {summary.total}")
    print(f"Latency avg/max  : {summary.avg_latency_ms:.2f} / {summary.max_latency_ms:.2f} ms")
    print(f"Within target    : {summary.within_target_ratio:.0%} (< 100 ms)")
    print(f"Avg confidence   : {summary.avg_confidence:.1%}")
    print(f"Fallbacks fired  : {summary.fallback_count}")

## 5. Next steps

- Full interactive console demo: `python main.py`
- Activation steps for Strands Decider 2B and Amazon Bedrock: see the repository `README.md`.
- Reusable pattern and its documentation: the `decision_patterns/` folder.